# 04 - Custom DMD

This notebook is the short path for your own data after you have seen the workflow in `03_build_your_own_animal.ipynb`, `01_intro_dmd.ipynb`, and `02_bird_flight_dmd.ipynb`.

Bring a motion array shaped `(frames, markers, dimensions)` and matching `times`. If you made a skeleton in notebook `03`, you can animate the raw data and DMD reconstruction. If you only have a NumPy array, skip the animation cells and use the 2D trace plots.

## Setup

Run this once at the start of Colab. If you are working locally in the repository environment, you can skip the install cell and run the imports.

In [ ]:
# Run this once at the start of Colab.
!pip install -q uv
!uv pip install --system -q git+https://github.com/LydiaFrance/dmd-workshop.git@main

from animal_dmd import setup_workshop
setup_workshop();

In [ ]:
import warnings

import numpy as np
from birddmd import reconstruct, run_dmd
from IPython.display import display

from morphing_birds import animate_plotly, animate_plotly_compare

from animal_dmd import (
    default_custom_skeleton_path,
    load_animal_bundle,
    expand_analysis_motion,
    load_custom_skeleton,
    plot_dmd_pair_traces,
    plot_marker_traces,
    plot_reconstruction_marker_traces,
    plot_unit_circle_eigenvalues,
    prepare_analysis_motion,
    prepare_custom_motion,
    print_dmd_summary,
)

np.set_printoptions(precision=3, suppress=True)
warnings.filterwarnings(
    "ignore",
    message="Input data condition number inf.*",
    category=UserWarning,
)

## Load your data

The easy path: run the cell below and upload the `my_animal.npz` bundle you
exported from notebook `03`. It restores your motion, timestamps, marker names,
and skeleton in one step.

If you did not use notebook `03`, use **Option B** in the cell to point at your
own arrays instead:

- `motion`: shape `(frames, markers, dimensions)`. A plain 2D matrix
  `(frames, features)` is treated as one marker per feature.
- `times`: shape `(frames,)`, in seconds.
- `marker_names` (optional): one name per marker; generated if omitted.
- `skeleton_path` (optional): leave as `None` if you have no skeleton.

In [ ]:
# --- Easiest path: load the bundle you exported from notebook 03 ---
bundle = load_animal_bundle()       # in Colab this prompts you to upload my_animal.npz
motion = bundle.motion
times = bundle.times
marker_names = bundle.marker_names
skeleton_path = bundle.skeleton_path

# --- Option B: bring your own arrays instead (comment out the block above) ---
# motion = np.load("my_marker_motion.npy")   # (frames, markers, dimensions)
# times = np.arange(motion.shape[0]) / 200   # seconds; use your own frame rate
# marker_names = None                         # or ["nose", "left_wing", ...]
# skeleton_path = None                        # no skeleton -> skip the animation cells

if motion is None or times is None:
    raise ValueError("Load your animal bundle, or fill in Option B with your own data.")

In [ ]:
prepared = prepare_custom_motion(motion, times, marker_names=marker_names)

motion = prepared.motion
times = prepared.times
marker_names = prepared.marker_names
dimension_labels = prepared.dimension_labels
mean_pose = prepared.mean_pose
centred_motion = prepared.centred_motion
data_matrix = prepared.data_matrix
dt = prepared.dt
n_frames = prepared.n_frames
n_markers = prepared.n_markers
n_dimensions = prepared.n_dimensions

full_motion = motion
full_marker_names = marker_names
full_mean_pose = mean_pose

## Optional: Load a Skeleton from Notebook 00

If you saved `custom_skeleton.json` from notebook `03`, this cell recreates the skeleton and attaches it to your current `motion` mean pose. In Colab, upload the JSON file if the runtime cannot find it.

In [ ]:
custom_skeleton = load_custom_skeleton(
    skeleton_path,
    full_motion,
    full_marker_names,
    full_mean_pose,
)
animal = custom_skeleton.animal
skel = custom_skeleton.skeleton
full_marker_names = custom_skeleton.marker_names

# DMD should use only analysis markers. If the skeleton has display-only
# markers, this removes them from the fitted data while keeping them for animation.
analysis_prepared = prepare_analysis_motion(
    full_motion,
    times,
    full_marker_names,
    animal=animal,
    analysis_marker_names=custom_skeleton.analysis_marker_names,
)

motion = analysis_prepared.motion
marker_names = analysis_prepared.marker_names
dimension_labels = analysis_prepared.dimension_labels
mean_pose = analysis_prepared.mean_pose
centred_motion = analysis_prepared.centred_motion
data_matrix = analysis_prepared.data_matrix
n_frames = analysis_prepared.n_frames
n_markers = analysis_prepared.n_markers
n_dimensions = analysis_prepared.n_dimensions

## Choose a Trace to Watch

DMD fits the whole motion, but a single marker coordinate is the simplest reconstruction check. Pick one marker and one dimension here. If you loaded a 2D feature matrix, use `selected_dimension = 0`.

In [ ]:
selected_marker = marker_names[0]
selected_dimension = min(2, n_dimensions - 1)

selected_marker_index = marker_names.index(selected_marker)
trace_label = f"{selected_marker} {dimension_labels[selected_dimension]}"

plot_marker_traces(
    times,
    {trace_label: motion[:, selected_marker_index, selected_dimension]},
    title=f"Input trace: {trace_label}",
)

## Optional: Animate the Raw Motion

Run this only if the skeleton loader created `animal`. Without a skeleton, continue to the DMD section.

In [ ]:
if animal is None:
    print("No skeleton provided, skipping animation.")
else:
    display(animate_plotly(animal, keypoints_frames=full_motion, axes_visible=True))

## Run DMD

Choose a small rank first. Increase it if the reconstruction misses important motion; decrease it if the fit looks noisy or hard to interpret.

In [ ]:
dmd_rank = min(6, n_frames - 1, n_markers * n_dimensions)

dmd_result = run_dmd(
    data=motion,
    times=times,
    n_modes=dmd_rank,
    d=2,
    eig_constraints={"conjugate_pairs"},
    average_shape=mean_pose,
    n_markers=n_markers,
    verbose=False,
)

dmd_reconstruction = dmd_result.reconstruction
reconstruction_times = times[1:]
reconstruction_rmse = np.sqrt(np.mean((motion[1:] - dmd_reconstruction) ** 2))

growth_per_second = dmd_result.eigenvalues.real
frequency_hz = dmd_result.eigenvalues.imag / (2 * np.pi)

print_dmd_summary(
    growth_per_second,
    frequency_hz,
    reconstruction_rmse,
    label=f"rank-{dmd_rank} reconstruction",
)

## Check the Reconstruction

Before interpreting modes, check whether the full DMD reconstruction matches the data you fitted. The reconstruction starts at `times[1]` because DMD learns how one frame advances to the next.

In [ ]:
plot_reconstruction_marker_traces(
    reconstruction_times,
    {trace_label: motion[1:, selected_marker_index, selected_dimension]},
    {trace_label: dmd_reconstruction[:, selected_marker_index, selected_dimension]},
    title=f"DMD reconstruction check: {trace_label}",
)

In [ ]:
per_frame_eigenvalues = np.exp(dmd_result.eigenvalues * dt)
plot_unit_circle_eigenvalues(per_frame_eigenvalues)

## Inspect DMD Mode Pairs

Oscillatory motions appear as conjugate pairs, one for each frequency. First list the pair frequencies, then choose the pairs you want to inspect.

In [ ]:
pair_frequencies_hz = np.array(
    [dmd_result.pair_frequency(pair_index) for pair_index in range(dmd_result.n_pairs)]
)

for pair_index, pair_frequency in enumerate(pair_frequencies_hz):
    print(f"pair {pair_index}: {pair_frequency:.2f} Hz")

In [ ]:
selected_pairs = list(range(min(3, dmd_result.n_pairs)))
# After looking at the printed frequencies, edit this list if you want specific pairs:
# selected_pairs = [0, 2]

pair_reconstructions = {
    pair_index: reconstruct(dmd_result, pairs=[pair_index])
    for pair_index in selected_pairs
}
pair_traces = {
    pair_index: pair_motion[:, selected_marker_index, selected_dimension]
    for pair_index, pair_motion in pair_reconstructions.items()
}

plot_dmd_pair_traces(
    reconstruction_times,
    motion[1:, selected_marker_index, selected_dimension],
    pair_traces,
    pair_frequencies_hz=pair_frequencies_hz,
    trace_label=trace_label,
)

## Optional: Animate the Reconstruction and Selected Pairs

Run these cells only if you provided a skeleton. For plain NumPy data, the trace plots above are the inspection view.

In [ ]:
if animal is None:
    print("No skeleton provided, skipping reconstruction animation.")
else:
    display(
        animate_plotly_compare(
            animal,
            keypoints_frames_list=[
                full_motion[1:],
                expand_analysis_motion(dmd_reconstruction, full_motion[1:], full_marker_names, marker_names),
            ],
            colours=["black", "crimson"],
            names=["data", "DMD reconstruction"],
            axes_visible=False,
        )
    )

In [ ]:
if animal is None:
    print("No skeleton provided, skipping pair animation.")
elif not pair_reconstructions:
    print("No DMD pairs selected.")
else:
    colours = ["royalblue", "darkorange", "seagreen", "purple", "crimson"]
    display(
        animate_plotly_compare(
            animal,
            keypoints_frames_list=[
                expand_analysis_motion(pair_motion, full_motion[1:], full_marker_names, marker_names)
                for pair_motion in pair_reconstructions.values()
            ],
            colours=colours[: len(pair_reconstructions)],
            names=[f"pair {pair_index}" for pair_index in pair_reconstructions],
            axes_visible=False,
        )
    )

## What to Adjust Next

- If the reconstruction misses obvious motion, try increasing `dmd_rank`.
- If the mode traces look noisy or hard to interpret, try decreasing `dmd_rank` or analysing a cleaner time window.
- If you have a skeleton and animation fails, check that `motion[:, i, :]` matches the same marker order expected by your `Animal3D` object.
- For biological interpretation, match pair frequencies to known behaviours or marker FFTs, then watch the pair animation to confirm the motion makes sense.